In [14]:
!pip install numpy
import numpy as np
import random as rd


[notice] A new release of pip is available: 25.3 -> 26.1
[notice] To update, run: pip install --upgrade pip


In [38]:
def simulation_complete_1player(rolls, Ns, verbose=False, heuristic=1):
    progress = {s: 0 for s in range(2, 13)}
    turn_idx = 0
    n_turns = 0

    HEURISTICS = {
        1: heuristic_random_stop4,
        2: rule_of_28,
        3: make_max_steps_stop_k(1),
        4: make_max_steps_stop_k(2),
        5: make_max_steps_stop_k(3),
        6: make_max_steps_stop_k(4),
        7: make_max_steps_stop_k(5),
        }

    if heuristic not in HEURISTICS:
        raise ValueError(f"Unknown heuristic: {heuristic}")

    heuristic_fn = HEURISTICS[heuristic]

    while nb_won_columns(progress, Ns) < 3:
        if turn_idx >= len(rolls):
            raise ValueError("Not enough turn-roll lists in rolls to finish the simulation.")

        temp_progress = {}
        is_turn_over = False
        turn_roll_count = 0
        turn_rolls = rolls[turn_idx]
        roll_idx = 0

        while not is_turn_over:
            if roll_idx >= len(turn_rolls):
                raise ValueError(
                    f"Not enough dice rolls in rolls[{turn_idx}] to finish this turn."
                )

            roll = tuple(int(x) for x in turn_rolls[roll_idx])
            roll_idx += 1
            turn_roll_count += 1

            pairings = get_pairings(roll)

            legal_actions = [
                action for action in pairings
                if action_is_legal(action, progress, temp_progress, Ns)
            ]

            if not legal_actions:
                temp_progress = {}
                is_turn_over = True
                continue

            action, stop = heuristic_fn(
                progress,
                temp_progress,
                legal_actions,
                Ns,
                turn_roll_count
            )

            temp_progress = apply_action(action, progress, temp_progress, Ns)

            if stop:
                progress = bank_progress(progress, temp_progress, Ns)
                is_turn_over = True

        if verbose:
            status = "BUST" if temp_progress == {} else "STOP"
            print(
                f"Turn {n_turns + 1:03d} | "
                f"rolls={turn_roll_count:02d} | "
                f"status={status} | "
                f"secured={progress}"
            )
        n_turns += 1
        turn_idx += 1

    return n_turns, progress

In [16]:
def action_is_legal(action, progress, temp_progress, Ns):
    new_columns = set()

    for s in action:
        current_pos = progress[s] + temp_progress.get(s, 0)

        if current_pos >= Ns[s]:
            return False

        if s not in temp_progress:
            new_columns.add(s)

    return len(temp_progress) + len(new_columns) <= 3



In [17]:
def apply_action(action, progress, temp_progress, Ns):
    """
    Returns a new temp_progress after applying the action.
    """
    new_temp = temp_progress.copy()

    for s in action:
        current_pos = progress[s] + new_temp.get(s, 0)

        if current_pos < Ns[s]:
            new_temp[s] = new_temp.get(s, 0) + 1

            # cap so we never go beyond the top
            if progress[s] + new_temp[s] > Ns[s]:
                new_temp[s] = Ns[s] - progress[s]

    return new_temp

In [18]:
def bank_progress(progress, temp_progress, Ns):
    new_progress = progress.copy()

    for s, inc in temp_progress.items():
        new_progress[s] = min(new_progress[s] + inc, Ns[s])

    return new_progress

In [19]:
def nb_won_columns(progress, Ns):
    return sum(progress[s] >= Ns[s] for s in range(2, 13))

In [20]:
def should_stop(progress, temp_progress, Ns, stop_threshold=3):
    total_temp = sum(temp_progress.values())
    return total_temp >= stop_threshold

In [21]:
def get_pairings(roll):
    d1, d2, d3, d4 = roll

    pairings = [
        tuple(sorted((d1 + d2, d3 + d4))),
        tuple(sorted((d1 + d3, d2 + d4))),
        tuple(sorted((d1 + d4, d2 + d3))),
    ]

    # remove duplicates while preserving order
    unique_pairings = []
    seen = set()
    for p in pairings:
        if p not in seen:
            seen.add(p)
            unique_pairings.append(p)

    return unique_pairings

In [42]:
Ns = {
    2: 3,
    3: 5,
    4: 7,
    5: 9,
    6: 11,
    7: 13,
    8: 11,
    9: 9,
    10: 7,
    11: 5,
    12: 3
}
N_games = 100
N_turns_max = 500
N_rolls_per_turn = 200

rng = np.random.default_rng(0)

rolls = [
    [
        [tuple(rng.integers(1, 7, size=4)) for _ in range(N_rolls_per_turn)]
        for _ in range(N_turns_max)
    ]
    for _ in range(N_games)
]

In [23]:
def evaluate_heuristic(rolls, Ns, heuristic, outlier_threshold=150):
    scores = []
    eliminated = 0

    for game_rolls in rolls:
        try:
            n_turns, progress = simulation_complete_1player(
                game_rolls,
                Ns,
                verbose=False,
                heuristic=heuristic
            )

            # outlier : partie beaucoup trop longue
            if n_turns > outlier_threshold:
                eliminated += 1
            else:
                scores.append(n_turns)

        except ValueError:
            # par exemple si pas assez de rolls pour finir
            eliminated += 1

    mean_score = np.mean(scores) if scores else None

    return mean_score, eliminated, len(scores)

In [24]:
def heuristic_random_stop4(progress, temp_progress, legal_actions, Ns, turn_roll_count):
    """
    heuristic with random pairing & stop after 4 turns 
    """
    # random choice of pairing
    action = rd.choice(legal_actions)

    # stops after 4 turns
    stop = (turn_roll_count >= 4)

    return action, stop

In [25]:
def rule_of_28(progress, temp_progress, legal_actions, Ns, turn_roll_count):
    action = choose_action_rule_of_28(progress, temp_progress, legal_actions, Ns)
    new_temp_progress = apply_action(action, progress, temp_progress, Ns) # should decide whether or not to stop after playing the next game
    stop = should_stop_rule_of_28(new_temp_progress)
    return action, stop

def choose_action_rule_of_28(progress, temp_progress, legal_actions, Ns):
    """
    legal_actions: list of actions like (5, 7) or (8, 8)
    progress[s]: secured progress in column s
    temp_progress[s]: temporary progress in current turn
    """

    def column_move_weight(s):
        # 6 - |7 - s| : favors middle columns
        return 6 - abs(7 - s)

    def move_value(action):
        value = 0

        # Count how many times each column is advanced by this action
        # e.g. (7,7) -> advances column 7 twice
        counts = {}
        for s in action:
            counts[s] = counts.get(s, 0) + 1

        for s, p_i in counts.items():
            # marker(i) = 1 if a new neutral marker is opened in column s
            opens_new_marker = int(s not in temp_progress)

            value += p_i * column_move_weight(s)
            value -= 6 * opens_new_marker

        return value

    return max(legal_actions, key=move_value)

def should_stop_rule_of_28(temp_progress):
    """
    Stop if progress value >= 28, as described in the paper.
    temp_progress[s] = number of spaces advanced this turn in column s
    """

    if not temp_progress:
        return False

    # base progress value: sum_i (s_i + 1)(|7 - i| + 1)
    value = 0
    open_columns = list(temp_progress.keys())

    for s, s_i in temp_progress.items():
        value += (s_i + 1) * (abs(7 - s) + 1)

    # difficulty adjustments only matter when 3 neutral markers are open
    if len(open_columns) == 3:
        all_odd = all(s % 2 == 1 for s in open_columns)
        all_even = all(s % 2 == 0 for s in open_columns)
        all_high = all(s >= 7 for s in open_columns)
        all_low = all(s <= 7 for s in open_columns)

        if all_odd:
            value += 2
        if all_even:
            value -= 2
        if all_high:
            value += 4
        if all_low:
            value += 4

    return value >= 28

In [26]:
def make_max_steps_stop_k(k):
    def heuristic(progress, temp_progress, legal_actions, Ns, turn_roll_count):
        def score(action):
            val = 0
            for s in action:
                val += 6 - abs(7 - s)   # favorise les colonnes centrales

                # bonus si la colonne est proche d'être finie
                current_pos = progress[s] + temp_progress.get(s, 0)
                remaining = Ns[s] - current_pos
                if remaining <= 2:
                    val += 3

                # petit malus si on ouvre une nouvelle colonne
                if s not in temp_progress:
                    val -= 1

            return val

        action = max(legal_actions, key=score)
        stop = (turn_roll_count >= k)
        return action, stop
    return heuristic

In [37]:
mean_score, eliminated, kept = evaluate_heuristic(
    rolls,
    Ns,
    heuristic=5,
    outlier_threshold=199
)

print("Mean score:", mean_score)
print("Eliminated games:", eliminated)
print("Kept games:", kept)

Mean score: 58.1
Eliminated games: 0
Kept games: 50


In [ ]:
for h in range(1, 8):
    mean_score, eliminated, kept = evaluate_heuristic(
        rolls,
        Ns,
        heuristic=h,
        outlier_threshold=199
    )

    print(f"Heuristic {h}: mean={mean_score}, eliminated={eliminated}, kept={kept}")

Heuristic 1: mean=111.42592592592592, eliminated=46, kept=54
Heuristic 2: mean=71.08, eliminated=0, kept=100
Heuristic 3: mean=27.65, eliminated=0, kept=100
Heuristic 4: mean=20.55, eliminated=0, kept=100
Heuristic 5: mean=59.41414141414141, eliminated=1, kept=99
Heuristic 6: mean=111.35416666666667, eliminated=52, kept=48
Heuristic 7: mean=148.55555555555554, eliminated=91, kept=9


: 

In [40]:
n_turns, progress = simulation_complete_1player(
    rolls[0],
    Ns,
    verbose=True,
    heuristic=1
)

print(n_turns)
print(progress)

Turn 001 | rolls=02 | status=BUST | secured={2: 0, 3: 0, 4: 0, 5: 0, 6: 0, 7: 0, 8: 0, 9: 0, 10: 0, 11: 0, 12: 0}
Turn 002 | rolls=02 | status=BUST | secured={2: 0, 3: 0, 4: 0, 5: 0, 6: 0, 7: 0, 8: 0, 9: 0, 10: 0, 11: 0, 12: 0}
Turn 003 | rolls=04 | status=STOP | secured={2: 0, 3: 0, 4: 2, 5: 0, 6: 0, 7: 4, 8: 0, 9: 0, 10: 2, 11: 0, 12: 0}
Turn 004 | rolls=03 | status=BUST | secured={2: 0, 3: 0, 4: 2, 5: 0, 6: 0, 7: 4, 8: 0, 9: 0, 10: 2, 11: 0, 12: 0}
Turn 005 | rolls=04 | status=BUST | secured={2: 0, 3: 0, 4: 2, 5: 0, 6: 0, 7: 4, 8: 0, 9: 0, 10: 2, 11: 0, 12: 0}
Turn 006 | rolls=04 | status=STOP | secured={2: 0, 3: 0, 4: 2, 5: 2, 6: 0, 7: 8, 8: 0, 9: 2, 10: 2, 11: 0, 12: 0}
Turn 007 | rolls=03 | status=BUST | secured={2: 0, 3: 0, 4: 2, 5: 2, 6: 0, 7: 8, 8: 0, 9: 2, 10: 2, 11: 0, 12: 0}
Turn 008 | rolls=02 | status=BUST | secured={2: 0, 3: 0, 4: 2, 5: 2, 6: 0, 7: 8, 8: 0, 9: 2, 10: 2, 11: 0, 12: 0}
Turn 009 | rolls=03 | status=BUST | secured={2: 0, 3: 0, 4: 2, 5: 2, 6: 0, 7: 8, 8: 0, 9